# Generate teacher-requested DWI QC products

This notebook creates only QC derivatives that are not already guaranteed by the preprocessing scripts. It does not choose a better pipeline. Run one subject at a time, beginning with D0007.

Already produced by v2: noise map, denoising residual, corrected mean b0, brain mask, FA, MD, AD, and RD. The cells below skip existing products unless `OVERWRITE=True`.

## 1. Configuration

Confirm the v1 directory. The v2 path matches the revised script shared on 2026-09-02.

In [9]:
from pathlib import Path
import csv, json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SUBJECT = 'D0007'
V1_DIR = (
    Path("/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc")
    / SUBJECT
)
V2_DIR = Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2') / SUBJECT
QC_ROOT = Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc') / SUBJECT
OVERWRITE = False

QC_V1 = QC_ROOT / 'v1'
QC_V2 = QC_ROOT / 'v2'
QC_COMPARE = QC_ROOT / 'v1_vs_v2'
for directory in (QC_V1, QC_V2, QC_COMPARE):
    directory.mkdir(parents=True, exist_ok=True)

V1 = {
    "raw": V1_DIR / f"{SUBJECT}_01_dwi_raw.mif",
    "denoised": V1_DIR / f"{SUBJECT}_02_dwi_denoised.mif",
    "noise": (
        V1_DIR
        / "qc"
        / f"{SUBJECT}_noise.mif"
    ),
    "residual": (
        V1_DIR
        / "qc"
        / f"{SUBJECT}_denoising_residual.mif"
    ),
    "degibbs": (
        V1_DIR
        / f"{SUBJECT}_03_dwi_degibbs.mif"
    ),
    "preprocessed": (
        V1_DIR
        / f"{SUBJECT}_04_dwi_preprocessed.mif"
    ),
    "corrected_b0": (
        V1_DIR
        / f"{SUBJECT}_corrected_mean_b0.nii.gz"
    ),
    "eddy_mask": (
        V1_DIR
        / f"{SUBJECT}_eddy_mask.nii.gz"
    ),
    "biascorrected": (
        V1_DIR
        / f"{SUBJECT}_05_dwi_biascorrected.mif"
    ),
    "tensor": (
        V1_DIR
        / f"{SUBJECT}_tensor.mif"
    ),
    "fa": V1_DIR / f"{SUBJECT}_fa.nii.gz",
    "md": V1_DIR / f"{SUBJECT}_md.nii.gz",
    "ad": V1_DIR / f"{SUBJECT}_ad.nii.gz",
    "rd": V1_DIR / f"{SUBJECT}_rd.nii.gz",
    "eddy_scratch": V1_DIR / "eddy_scratch",
}
V2 = {
    'raw': V2_DIR / f'{SUBJECT}_01_raw.mif',
    'denoised': V2_DIR / f'{SUBJECT}_02_denoised.mif',
    'noise': V2_DIR / 'qc' / f'{SUBJECT}_noise.mif',
    'residual': V2_DIR / 'qc' / f'{SUBJECT}_denoising_residual.mif',
    'degibbs': V2_DIR / f'{SUBJECT}_03_degibbs.mif',
    'preprocessed': V2_DIR / f'{SUBJECT}_04_preprocessed.mif',
    'corrected_b0': V2_DIR / f'{SUBJECT}_corrected_mean_b0.nii.gz',
    'mask': V2_DIR / f'{SUBJECT}_corrected_mean_b0_brain_mask.nii.gz',
    'biascorrected': V2_DIR / f'{SUBJECT}_05_biascorrected.mif',
    'fa': V2_DIR / f'{SUBJECT}_fa.nii.gz',
    'md': V2_DIR / f'{SUBJECT}_md.nii.gz',
    'ad': V2_DIR / f'{SUBJECT}_ad.nii.gz',
    'rd': V2_DIR / f'{SUBJECT}_rd.nii.gz',
    'eddy_scratch': V2_DIR / 'eddy_scratch',
}
print('QC output:', QC_ROOT)

QC output: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007


In [10]:
def run_product(command, output):
    output = Path(output)
    if output.exists() and not OVERWRITE:
        print('Exists, skipped:', output)
        return True
    missing_tools = [command[0]] if shutil.which(command[0]) is None else []
    if missing_tools:
        print('Missing command:', missing_tools[0])
        return False
    print('Running:', ' '.join(map(str, command)))
    subprocess.run([str(x) for x in command], check=True)
    return output.exists()

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

## 2. Denoising residuals

Teacher-requested product: `raw - denoised`. v2 already creates it; this cell creates the missing v1 residual. A noise map cannot be reconstructed by subtraction; v1 must rerun `dwidenoise -noise` if a v1 noise map is desired.

In [11]:
v1_residual = QC_V1 / f'{SUBJECT}_denoising_residual.mif'
require(V1['raw'], V1['denoised'])
run_product(['mrcalc', V1['raw'], V1['denoised'], '-subtract', v1_residual, '-force'], v1_residual)
print('v2 residual:', V2['residual'], 'exists =', V2['residual'].exists())

Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_denoising_residual.mif
v2 residual: /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/qc/D0007_denoising_residual.mif exists = True


## 3. Matching 3D images for Gibbs inspection

Teacher-requested product: matched denoised and degibbs volumes for side-by-side inspection. Set `DWI_VOLUME` to the same diffusion volume for both pipelines.

In [12]:
DWI_VOLUME = 2
for label, source in [('v1_denoised', V1['denoised']), ('v1_degibbs', V1['degibbs']), ('v2_denoised', V2['denoised']), ('v2_degibbs', V2['degibbs'])]:
    destination = (QC_V1 if label.startswith('v1') else QC_V2) / f'{SUBJECT}_{label}_vol{DWI_VOLUME}.nii.gz'
    require(source)
    run_product(['mrconvert', source, destination, '-coord', '3', str(DWI_VOLUME), '-force'], destination)

Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_v1_denoised_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_v1_degibbs_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v2/D0007_v2_denoised_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v2/D0007_v2_degibbs_vol2.nii.gz


## 4. Corrected mean b0 for v1

Teacher-requested comparison product: v1 corrected mean b0. v2 already saves the equivalent image.

In [13]:
v1_corrected_b0 = QC_V1 / f'{SUBJECT}_corrected_mean_b0.nii.gz'
require(V1['preprocessed'])
if not v1_corrected_b0.exists() or OVERWRITE:
    command = f"dwiextract '{V1['preprocessed']}' -bzero - | mrmath - mean '{v1_corrected_b0}' -axis 3 -force"
    subprocess.run(command, shell=True, executable='/bin/bash', check=True)
print('v1 corrected b0:', v1_corrected_b0)
print('v2 corrected b0:', V2['corrected_b0'])

v1 corrected b0: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_corrected_mean_b0.nii.gz
v2 corrected b0: /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/D0007_corrected_mean_b0.nii.gz


## 5. Eddy motion, outliers, and eddy_quad inputs

Teacher-requested products. v2 preserves `eddy_scratch`; v1 did not use `-nocleanup`, so old v1 runs may not contain enough files. This inventory identifies available eddy files without guessing their names. If v1 files are absent, v1 must be rerun while preserving its eddy working directory; the products cannot be recovered from the final DWI alone.

In [14]:
patterns = ['*parameter*', '*movement*', '*outlier*', '*cnr*', '*residual*', '*eddy*', '*acqp*', '*index*']
def eddy_inventory(root, pipeline):
    root = Path(root)
    rows = []
    if root.exists():
        seen = set()
        for pattern in patterns:
            for path in root.rglob(pattern):
                if path.is_file() and path not in seen:
                    seen.add(path)
                    rows.append({'pipeline': pipeline, 'file': str(path), 'size_bytes': path.stat().st_size})
    return rows

eddy_files = pd.DataFrame(eddy_inventory(V1_DIR, 'v1') + eddy_inventory(V2['eddy_scratch'], 'v2'))
eddy_inventory_csv = QC_COMPARE / f'{SUBJECT}_eddy_file_inventory.csv'
eddy_files.to_csv(eddy_inventory_csv, index=False)
display(eddy_files)
print('Saved:', eddy_inventory_csv)

,pipeline,file,size_bytes
0,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,6719
1,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,786
2,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,425
3,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,6780
4,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,763
5,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,753
6,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,28581
7,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0
8,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,6682
9,v1,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,28901


Saved: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1_vs_v2/D0007_eddy_file_inventory.csv


In [15]:
key_eddy_files = eddy_files[
    eddy_files["file"].str.contains(
        "parameter|movement|outlier|cnr|residual|index|config|mask|bval",
        case=False,
        regex=True,
    )
]

print(
    key_eddy_files[
        ["pipeline", "file", "size_bytes"]
    ].to_string(index=False)
)

pipeline                                                                                                                                                                 file  size_bytes
      v1                                /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/eddy_scratch/dwifslpreproc-tmp-E573QZ/dwi_post_eddy.eddy_parameters        6719
      v1      /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/eddy_scratch/dwifslpreproc-tmp-E573QZ/dwi_post_eddy.eddy_post_eddy_shell_alignment_parameters         786
      v1 /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/eddy_scratch/dwifslpreproc-tmp-E573QZ/dwi_post_eddy.eddy_post_eddy_shell_PE_translation_parameters         425
      v1            /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/eddy_scratch/dwifslpreproc-tmp-E573QZ/dwi_post_eddy.eddy_values_of_all_input_parameters        6780
      v1                   /mnt/newStor/paros/paros_WORK/xintong/test/

In [16]:
def summarize_eddy_outlier_map(path):
    # FSL eddy_outlier_map contains one descriptive text header line.
    values = np.loadtxt(path, skiprows=1)
    values = np.atleast_2d(values)
    flagged = values != 0
    return {
        'file': str(path),
        'n_flagged': int(flagged.sum()),
        'n_evaluated': int(flagged.size),
        'outlier_percent': float(100 * flagged.mean()),
    }

candidate_maps = []
if not eddy_files.empty:
    candidate_maps = [Path(p) for p in eddy_files['file'] if 'outlier_map' in Path(p).name and 'n_stdev' not in Path(p).name]
outlier_summary = pd.DataFrame([summarize_eddy_outlier_map(p) for p in candidate_maps])
outlier_csv = QC_COMPARE / f'{SUBJECT}_eddy_outlier_summary.csv'
outlier_summary.to_csv(outlier_csv, index=False)
display(outlier_summary)
print('Saved:', outlier_csv)
print('Run eddy_quad only after its eddy basename, index, acqp, mask, and bvals are confirmed in the inventory.')

ValueError: could not convert string 'One' to float64 at row 0, column 1.

In [ ]:
# Fill these from the inventory for one pipeline, then run eddy_quad.
EDDY_PIPELINE = 'v2'
EDDY_BASE = None       # Path without suffix, e.g. .../dwi_post_eddy
EDDY_INDEX = None      # eddy index file
EDDY_ACQPARAMS = None  # acquisition-parameter file
EDDY_MASK = None       # eddy mask
EDDY_BVALS = None      # b-values used by eddy

if any(x is None for x in (EDDY_BASE, EDDY_INDEX, EDDY_ACQPARAMS, EDDY_MASK, EDDY_BVALS)):
    print('Skipped eddy_quad: confirm all five paths from the eddy inventory first.')
else:
    eddy_quad_dir = QC_ROOT / EDDY_PIPELINE / 'eddy_quad'
    eddy_quad_dir.mkdir(parents=True, exist_ok=True)
    command = ['eddy_quad', str(EDDY_BASE), '-idx', str(EDDY_INDEX), '-par', str(EDDY_ACQPARAMS), '-m', str(EDDY_MASK), '-b', str(EDDY_BVALS), '-o', str(eddy_quad_dir)]
    subprocess.run(command, check=True)
    print('Saved eddy_quad report:', eddy_quad_dir)

## 6. Mask comparison products

Teacher-requested product: mask overlap summary and a disagreement mask. This requires v1 and v2 masks to be on the same voxel grid. Dice measures agreement, not anatomical correctness; the disagreement image must still be inspected on mean b0.

In [ ]:
V1_MASK = V1['eddy_mask']
V2_MASK = V2['mask']
mask_xor = QC_COMPARE / f'{SUBJECT}_mask_disagreement.nii.gz'
mask_stats = QC_COMPARE / f'{SUBJECT}_mask_overlap.csv'
require(V1_MASK, V2_MASK)
run_product(['mrcalc', V1_MASK, V2_MASK, '-xor', mask_xor, '-datatype', 'bit', '-force'], mask_xor)

def count_nonzero_voxels(path):
    result = subprocess.run(['mrstats', str(path), '-mask', str(path), '-output', 'count'], check=True, capture_output=True, text=True)
    return float(result.stdout.strip())

intersection = QC_COMPARE / f'{SUBJECT}_mask_intersection.nii.gz'
run_product(['mrcalc', V1_MASK, V2_MASK, '-mult', intersection, '-datatype', 'bit', '-force'], intersection)
n1, n2, ni = [count_nonzero_voxels(p) for p in (V1_MASK, V2_MASK, intersection)]
dice = 2 * ni / (n1 + n2) if (n1 + n2) else np.nan
pd.DataFrame([{'subject': SUBJECT, 'v1_voxels': n1, 'v2_voxels': n2, 'intersection_voxels': ni, 'dice': dice}]).to_csv(mask_stats, index=False)
display(pd.read_csv(mask_stats))

## 7. FA/MD/AD/RD difference maps

Teacher-requested comparison products. Configure v1 metric paths below. Images must already share the same grid; this notebook stops rather than silently resampling quantitative maps.

In [ ]:
V1_METRICS = {
    'fa': V1_DIR / f'{SUBJECT}_fa.nii.gz',
    'md': V1_DIR / f'{SUBJECT}_md.nii.gz',
    'ad': V1_DIR / f'{SUBJECT}_ad.nii.gz',
    'rd': V1_DIR / f'{SUBJECT}_rd.nii.gz',
}
V1_METRICS['md'] = V1_METRICS['md'] if V1_METRICS['md'].exists() else V1_DIR / f'{SUBJECT}_adc.nii.gz'
rows = []
for metric in ('fa', 'md', 'ad', 'rd'):
    old, new = V1_METRICS[metric], V2[metric]
    if not old.exists() or not new.exists():
        rows.append({'metric': metric.upper(), 'status': 'missing input', 'v1': str(old), 'v2': str(new)})
        continue
    difference = QC_COMPARE / f'{SUBJECT}_{metric}_v2_minus_v1.nii.gz'
    run_product(['mrcalc', new, old, '-subtract', difference, '-force'], difference)
    rows.append({'metric': metric.upper(), 'status': 'created', 'difference_map': str(difference)})
metric_products = pd.DataFrame(rows)
metric_products.to_csv(QC_COMPARE / f'{SUBJECT}_metric_product_status.csv', index=False)
display(metric_products)

## 8. Regional diffusion stability

Teacher-requested product, but it requires one atlas/label image already transformed into the diffusion grid. Set `ATLAS_IN_DWI` only after registration is verified. The output contains v1, v2, absolute difference, and percent difference for every nonzero label.

In [ ]:
ATLAS_IN_DWI = None  # Example: Path('/path/to/atlas_in_DWI_space.nii.gz')
if ATLAS_IN_DWI is None:
    print('Skipped: set ATLAS_IN_DWI after anatomical alignment is verified.')
else:
    atlas = Path(ATLAS_IN_DWI)
    require(atlas)
    regional_rows = []
    for metric in ('fa', 'md', 'ad', 'rd'):
        old, new = V1_METRICS[metric], V2[metric]
        require(old, new)
        labels_text = subprocess.run(['mrstats', str(atlas), '-output', 'max'], check=True, capture_output=True, text=True).stdout
        max_label = int(float(labels_text.strip()))
        for label in range(1, max_label + 1):
            temp_mask = QC_COMPARE / f'_tmp_label_{label}.mif'
            subprocess.run(['mrcalc', atlas, str(label), '-eq', temp_mask, '-force'], check=True)
            means = []
            for image in (old, new):
                value = subprocess.run(['mrstats', str(image), '-mask', str(temp_mask), '-output', 'mean'], check=True, capture_output=True, text=True)
                means.append(float(value.stdout.strip()))
            if np.isfinite(means).all():
                diff = means[1] - means[0]
                pct = 100 * diff / means[0] if means[0] != 0 else np.nan
                regional_rows.append({'subject': SUBJECT, 'metric': metric.upper(), 'label': label, 'v1_mean': means[0], 'v2_mean': means[1], 'difference': diff, 'percent_difference': pct})
            temp_mask.unlink(missing_ok=True)
    regional = pd.DataFrame(regional_rows)
    regional.to_csv(QC_COMPARE / f'{SUBJECT}_regional_diffusion_stability.csv', index=False)
    display(regional.head())

## Products that still require a preserved/re-run eddy job

The notebook cannot reconstruct motion parameters, model-fit residuals, or a valid eddy_quad report from the final preprocessed image. If the required files are absent from the inventory, rerun that pipeline while preserving/exporting eddy outputs. Anatomical-alignment products also require the confirmed DWI-to-T1 transform. Tractography/connectome reproducibility is explicitly deferred (`if needed later`).